# Sensor data cleaning – Storey 1 (v2, with provenance)

Cleans the CeDInt sensor workbook and produces one clean CSV per zone for the
forecasting model. Every change is recorded at **cell level**, so any value in the output can be
traced back to the rule that produced it and to its original value.

## What changed from v1

| | v1 | v2 |
|---|---|---|
| Column meaning | `W` reactive, `Pa` active | **`W` active (W), `Pa` apparent (VA), `Er` active energy, `Ea` apparent energy** |
| Forecasting target | `Ea` difference (apparent energy) | **`Er` difference (active energy)** |
| Negative `W` | corrected only when `Pa` or `I` also negative | always an error on a consuming load |
| Period split | gap > 30 days | gap > 90 days (a real 37-day outage sits inside era A) |
| Rolling windows | fixed row counts | fixed **time** spans, converted per sampling regime |
| Energy validation | counter vs power on all rows | **only rows where power was not used to build the value**, tested for bias, per regime |
| Change tracking | counts per rule | **one audit row per changed cell + a bitmask column per measurement column** |

## How the column meanings were established

On the two single-meter channels, `Pa / I` = 228.2 V and 231.4 V — mains voltage, so
`Pa = V x I` is apparent power. `ΔEa / W` equals `1 / power factor` exactly
(1NE HVAC: pf 0.215, ratio 4.649), while `ΔEr / W` = 0.998 across all series.
`W <= Pa` holds in 99.85% of rows and `Ea >= Er` in 99.71% — the P <= S and
apparent >= active bounds. This matches the course slide: *Apparent Energy (Ea),
Active Energy (Er)*.

## The rule for choosing a correction
1. **Invent nothing.** A wrong value becomes empty (NaN) rather than a made-up number.
   A value is rebuilt only where physics allows it.
2. **Lose nothing silently.** Every change is written to `audit.parquet` with the old
   and new value, and marked in a `*_flags` column in the output itself.

**Run with Runtime → Run all.**

## 0. Settings, rule codebook and the provenance engine

`CONFIG` holds every threshold. `RULES` is the codebook: each rule owns one bit of the
`*_flags` columns. `Fixer.apply` is the only function in the notebook that changes a
value — it records the old value, writes the new one, sets the bit and appends to the
audit table in one step, so the data and its log cannot drift apart.

In [ ]:
"""Core cleaning engine: config, provenance, the single mutation function."""
import re, json, hashlib, datetime as _dt
import numpy as np, pandas as pd

# ----------------------------------------------------------------- CONFIG
CONFIG = {
    "zone_pattern": r"(?:^|[^A-Z0-9])(NE|NW|SE|SW)(?:[^A-Z0-9]|$)",
    # 90 d: safely above the real 37-day outage inside era A (1NW Emerg. Lighting /
    # HVAC, Jun 2021) and far below the 901-day pause between the two campaigns.
    "new_period_gap_days": 90,
    "corrupted_x_median": 1000,
    "corrupted_x_p999": 20,
    "tukey_k": 3.0,
    "counter_tolerance": 200,
    "drop_return_hours": 24,
    "jump_margin": 3.0,
    "jump_confirm_readings": 10,
    "min_power_for_current_W": 5.0,
    "partial_ratio": 0.6,
    "temp_physical": (-10.0, 60.0),
    "hum_jump": 5.0,
    # windows and gaps are expressed in HOURS and converted to a row count per
    # period, so they mean the same span in the hourly and the per-minute data.
    "partial_ref_window_h": 3.0,
    "hum_median_window_h": 3.0,
    "max_interp_gap_h": 0.25,
    "max_power_energy_gap_h": 6,
}

# Column semantics, established by test against the data (see README):
#   W  = ACTIVE power      (W)    pairs with Er
#   Er = ACTIVE energy     (Wh)   <- the quantity to forecast
#   Pa = APPARENT power    (VA)   pairs with Ea
#   Ea = APPARENT energy   (VAh)
ELEC = ["Pa", "W", "I", "Ea", "Er"]
ENV = ["Lum", "Hum", "Temp", "Pres"]
COLUMNS = ["Timestamp", "W", "I", "Ea", "Er", "Pa", "E Device type",
           "Number of Edevices", "Lum", "Hum", "Temp", "Pres"]
FLAGGED = ELEC + ENV                      # columns that carry a *_flags column
FLAG_COLS = [c + "_flags" for c in FLAGGED]
EXTRA_COLS = FLAG_COLS + ["energy_Wh", "energy_source", "apparent_energy_VAh"]

# ----------------------------------------------------------------- RULES
# The codebook. Each rule owns one bit of the *_flags columns and carries its own
# justification, so rules.csv explains every correction without needing the notebook.
#   what      - the condition that triggers it
#   why       - why that reading cannot be right
#   action    - what the script does about it
#   rejected  - the alternatives considered and why they were not chosen
#   evidence  - the measurement that justifies the rule
#   config    - the CONFIG keys that control it
#   reversible- whether the original value can be recovered from the audit file
def _r(rule, what, why, action, rejected, evidence, config, reversible="yes"):
    return dict(rule=rule, what_detected=what, why_wrong=why, action=action,
                alternatives_rejected=rejected, evidence=evidence,
                config_keys=config, reversible=reversible)

RULES = {
 0: _r("dup_timestamp",
       "The same timestamp appears twice in a sheet, or the timestamp cannot be parsed",
       "A sensor reports once per instant, so a repeated timestamp is an export artefact",
       "Keep the first row, drop the rest",
       "Averaging the duplicates: would invent a reading that no meter produced",
       "Removed rows are counted per sensor in info['dropped_rows']",
       "-", "no - the row is not in the output"),
 1: _r("sensor_unmeasured",
       "Every electrical column is exactly 0 in every row of the sheet",
       "A cumulative counter that never moves in five years is a disconnected meter, not a device that is off",
       "Leave the sensor out of the output file entirely",
       "Keeping it as zeros: the model would learn five years of zero consumption as real",
       "SE Projector: 61,689 rows, all five electrical columns exactly 0",
       "-", "no - the sensor is not in the output"),
 2: _r("sign_inverted",
       "The median active power of the sensor is negative over the whole series",
       "Lighting and sockets consume energy, they do not generate it; the current clamp is fitted backwards",
       "Multiply W, I, Pa, Ea and Er by -1",
       "Deleting the sensor: would throw away 216,000 usable rows for a fault that is a sign flip",
       "NW Emerg. Lighting: 99.3% of rows negative and the counter runs backwards (153031 -> 152900 -> 152769)",
       "-", "yes"),
 3: _r("all_zero_reading",
       "Pa, W, I, Ea and Er are all exactly 0 in the same row",
       "A cumulative counter cannot return to zero; the whole reading is missing and was written as zeros",
       "Set all five to empty (NaN)",
       "Keeping the zeros: would create a huge negative then positive consumption in the counter difference",
       "Zeros occur at the same timestamps across sheets, next to time gaps",
       "-", "yes"),
 4: _r("counter_zero",
       "Ea or Er is exactly 0 in the middle of the series while power is normal",
       "A cumulative counter only increases; a 0 mid-series is a missing value written as zero",
       "Set the counter to empty (NaN)",
       "Interpolating immediately: the energy is recovered more accurately from the valid readings either side",
       "Zeros cluster around data gaps rather than at low-consumption periods",
       "-", "yes"),
 5: _r("counter_negative",
       "Ea or Er is below zero",
       "A cumulative energy counter has no physical meaning below zero",
       "Set the counter to empty (NaN)",
       "Absolute value: the magnitude of a corrupted counter carries no information",
       "82,101 negative counter cells across the four workbooks before cleaning",
       "-", "yes"),
 6: _r("corrupted_value",
       "A power or current value over 1,000x the typical magnitude, or 20x the 99.9th percentile",
       "A circuit drawing 900 W cannot report 21 million W",
       "Set the value to empty (NaN)",
       "Capping at the threshold: would invent a plausible-looking number in place of a known error",
       "NW Plugs 2020-10-05 12:00: Pa = 2.1e7 W; that single row drags the W-Pa correlation from 0.93 to 0.01",
       "corrupted_x_median, corrupted_x_p999", "yes"),
 7: _r("negative_recovered",
       "Pa, W or I is negative and its magnitude sits inside the sensor's normal range",
       "These circuits only consume, so a negative reading is a sign error on an otherwise valid measurement",
       "Replace with the absolute value",
       "Emptying it: discards a reading whose magnitude is clearly correct",
       "W is active power, so every negative value on a consuming load is an error, not a capacitive effect",
       "tukey_k", "yes"),
 8: _r("negative_nulled",
       "Pa, W or I is negative and its magnitude would be an extreme outlier (above Q3 + 3 IQR)",
       "Both the sign and the magnitude are wrong, so nothing in the reading can be trusted",
       "Set the value to empty (NaN)",
       "Absolute value: would turn a corrupted reading into a false peak",
       "Separating the two cases keeps 97,185 recoverable values while discarding only 99",
       "tukey_k", "yes"),
 9: _r("counter_partial",
       "The counter drops below its level and returns to it within 24 hours",
       "Each sheet sums up to 92 meters; when some do not answer, the total drops by their share and recovers when they return",
       "Empty the affected readings; the energy is recovered from the valid readings on either side, so none is lost",
       "Absolute difference: would invent millions of Wh; interpolation: would hide the dropout instead of recording it",
       "Negative counter steps fall from 12.0% to 0.1% after this rule; the share rises with meter count (Spearman 0.62)",
       "counter_tolerance, drop_return_hours", "yes"),
 10: _r("counter_spike",
       "An isolated counter value far from its neighbours that the following readings do not confirm",
       "A one-off jump with no lasting change is a transmission error, not a new meter level",
       "Set the counter to empty (NaN)",
       "Treating it as a jump: would shift the whole counter baseline on the strength of one reading",
       "Confirmed against the next 10 readings before deciding spike vs jump",
       "jump_margin, jump_confirm_readings", "yes"),
 11: _r("counter_reset",
       "The counter drops and never returns to its previous level",
       "The collector restored the counter to an earlier saved value; the reading is valid, the difference across it is not",
       "Keep the value as a new valid level, but do not use the step across it as consumption",
       "Emptying it: the value is correct and later readings build on it",
       "Resets hit up to 17 of 18 series at the same timestamp, so the cause is the platform, not the meters",
       "counter_tolerance", "yes - flagged, value unchanged"),
 12: _r("counter_jump",
       "The counter rises more than the maximum power allows and the following readings confirm the new level",
       "A genuine step change in the counter baseline, for example after a meter was replaced or reconfigured",
       "Keep the value as a new valid level, but do not use the step across it as consumption",
       "Emptying it: would discard a correct reading and break every later difference",
       "Confirmed only when the next 10 readings stay near the new level",
       "jump_margin, jump_confirm_readings, max_power_energy_gap_h", "yes - flagged, value unchanged"),
 13: _r("partial_power",
       "The counter is a partial sum in that row and the power is below 60% of its neighbours",
       "When only some meters answer, the power columns are a partial sum too, not only the counter",
       "Empty Pa, W and I for that row",
       "Keeping them: the model would see consumption collapsing for a minute and recovering, which never happened",
       "Two independent columns agreeing (counter dips and power dips together) is a far more reliable signal than either alone",
       "partial_ratio, partial_ref_window_h", "yes"),
 14: _r("zero_is_missing",
       "A zero contradicted by another column: Pa = 0 while Ea rises, W = 0 while Er rises, or I = 0 with Pa above 5 W",
       "If the energy counter advanced, the device was consuming, so the zero is a missing value",
       "Set the value to empty (NaN)",
       "Keeping all zeros: would teach the model switch-offs that never happened and lower every average",
       "NE HVAC: 89.9% of hourly rows have I = 0, and all of those have Pa above 5 W - a resolution floor, not a reading",
       "min_power_for_current_W", "yes"),
 15: _r("outlier_kept",
       "A high value (above Q3 + 3 IQR) confirmed by a second column or by the energy counter",
       "A peak that several independent measurements agree on is a real event, such as a compressor starting",
       "Keep the value, flag it only",
       "Removing or capping: real peaks are the most informative moments for a consumption model",
       "Confirmation requires a second column in the 99th percentile, or the counter to have recorded that energy",
       "tukey_k", "yes - flagged, value unchanged"),
 16: _r("outlier_nulled",
       "A high value that no other column in the same row supports",
       "One quantity alone being extreme, with current and energy normal, is a measurement error",
       "Set the value to empty (NaN)",
       "Winsorising at the threshold: would invent a number and erase the distinction from real peaks",
       "Only upward outliers are treated; low values are real night and weekend periods the model must learn",
       "tukey_k", "yes"),
 17: _r("env_sensor_absent",
       "An environmental column is 0 in every row of the zone",
       "A constant 0 degrees and 0% humidity indoors for five years means the sensor is absent or unconnected",
       "Empty the whole column for that zone",
       "Loading it as zeros: would produce an average temperature of 0 degrees with no warning",
       "Zone NE reports no environmental data at all: Lum, Hum, Temp and Pres are 0 in all 88,828 rows",
       "-", "yes"),
 18: _r("env_reading_missing",
       "Humidity and temperature are both exactly 0 in the same row, or either is exactly 0 alone",
       "0 degrees C and 0% relative humidity are impossible in an occupied building; the reading is missing",
       "Set the environmental values to empty (NaN)",
       "Keeping the zeros: would pull every environmental average towards zero",
       "508,949 zero cells across Temp and Hum before cleaning",
       "-", "yes"),
 19: _r("temp_impossible",
       "Temperature below -10 C or above 60 C",
       "Outside this range the reading cannot come from an indoor sensor",
       "Set the temperature to empty (NaN)",
       "A narrower 10-35 C band: would delete 7,935 genuine cold readings, 6,571 of them in January 2021 (storm Filomena) - exactly the cold weather an energy model needs",
       "SW records up to 1,411 C for one week, 1-7 March 2022: 534 rows, a contained sensor fault",
       "temp_physical", "yes"),
 20: _r("hum_isolated",
       "Humidity more than 5 points from the median of its neighbours over 3 hours, or outside 0-100%",
       "Indoor humidity changes slowly, so a single reading far from its neighbours is a sensor step",
       "Set the humidity to empty (NaN), then fill it if the gap is short",
       "Keeping it: single-reading spikes would be learned as real humidity events",
       "Hum has only 19 distinct integer values in 1.3M rows, so its resolution is already low",
       "hum_jump, hum_median_window_h", "yes"),
 21: _r("pres_not_binary",
       "Pres holds a value other than 0 or 1",
       "Pres is a presence detector, not the atmospheric pressure the column name suggests, so only 0 and 1 are valid",
       "Set the value to empty (NaN)",
       "Rescaling to a pressure: the column has only ever contained 0 and 1, so there is nothing to rescale",
       "At 1-minute resolution NW and SE show a clean occupancy curve (0.00 overnight, peak 0.238 at 11:00 UTC) with a median run of 2 minutes - a motion sensor, not a pressure",
       "-", "yes"),
 22: _r("interpolated",
       "A gap of at most 15 minutes in Pa, W, I, Hum or Temp",
       "These quantities change continuously, so a short gap can be bridged without inventing a trend",
       "Fill linearly between the valid readings either side",
       "Filling long gaps: would invent data; filling Lum or Pres: both change abruptly and Pres is binary",
       "The limit is set in hours and converted per sampling regime, so it means 15 minutes in both eras",
       "max_interp_gap_h", "yes"),
 23: _r("pf_implausible",
       "Active power exceeds apparent power (W > Pa)",
       "Active power cannot exceed apparent power for a single meter (P <= S)",
       "Flag the row, change nothing",
       "Correcting either column: the data cannot say which of the two is wrong, so any fix would be a guess",
       "0.52% of rows overall, concentrated in SE Lighting (7.8%); these columns are sums over up to 92 meters, so the per-meter bound need not hold on the total",
       "-", "yes - flagged, value unchanged"),
}
BIT = {v["rule"]: b for b, v in RULES.items()}
RULE_FIELDS = ["what_detected", "why_wrong", "action", "alternatives_rejected",
               "evidence", "config_keys", "reversible"]

def rules_table():
    return pd.DataFrame([{"bit": b, "flag_value": 1 << b, "rule": v["rule"],
                          **{f: v[f] for f in RULE_FIELDS}}
                         for b, v in sorted(RULES.items())])

# ----------------------------------------------------------------- RUN ID
def make_run_id(config):
    h = hashlib.sha1(json.dumps(config, sort_keys=True, default=str).encode()).hexdigest()[:8]
    return f"{_dt.datetime.now():%Y%m%dT%H%M%S}-{h}"

# ----------------------------------------------------------------- AUDIT
class Audit:
    """Collects every cell-level change. One row per changed cell."""
    def __init__(self, run_id, dry_run=False):
        self.run_id, self.dry_run, self._parts = run_id, dry_run, []

    def record(self, sensor, df, idx, col, rule, old, new):
        if len(idx) == 0:
            return
        self._parts.append(pd.DataFrame({
            "run_id": self.run_id, "sensor": sensor, "column": col, "rule": rule,
            "timestamp": df.loc[idx, "Timestamp"].to_numpy(),
            "excel_row": df.loc[idx, "excel_row"].to_numpy(),
            "old_value": np.asarray(old, dtype=float),
            "new_value": np.asarray(new, dtype=float)}))

    def frame(self):
        if not self._parts:
            return pd.DataFrame(columns=["run_id","sensor","column","rule","timestamp",
                                         "excel_row","old_value","new_value"])
        return pd.concat(self._parts, ignore_index=True)

    def summary(self):
        f = self.frame()
        if f.empty:
            return f
        return (f.groupby(["sensor", "rule", "column"]).size()
                 .rename("cells").reset_index().sort_values("cells", ascending=False))


# ----------------------------------------------------------------- THE MUTATION FUNCTION
class Fixer:
    """Every correction in the pipeline goes through .apply().

    It records the old values, writes the new ones, sets the provenance bit and
    appends to the audit table in one step, so the data and its log cannot drift
    apart. A cell is only recorded when its value actually changes.
    """
    def __init__(self, audit):
        self.audit = audit

    @staticmethod
    def init_flags(df, cols=None):
        for c in (cols or FLAGGED):
            if c in df.columns:
                df[c + "_flags"] = np.int64(0)
        return df

    def apply(self, sensor, df, mask, cols, rule, new=np.nan, note_only=False):
        """mask: boolean Series/array aligned to df. cols: column or list.
        new:   scalar, or a Series aligned to df (e.g. the absolute value).
        note_only=True sets the flag and logs without changing the value
        (used for 'kept' decisions such as a confirmed outlier)."""
        bit = BIT[rule]
        if isinstance(cols, str):
            cols = [cols]
        m = np.asarray(pd.Series(mask, index=df.index).fillna(False), dtype=bool)
        total = 0
        for c in cols:
            if c not in df.columns:
                continue
            old = df[c].to_numpy(dtype=float, copy=True)
            newv = (np.asarray(pd.Series(new, index=df.index), dtype=float)
                    if isinstance(new, (pd.Series, np.ndarray)) else np.full(len(df), float(new)))
            if note_only:
                changed = m
            else:
                # a cell changes only if the value really differs (NaN->NaN is not a change)
                differs = ~((np.isnan(old) & np.isnan(newv)) | (old == newv))
                changed = m & differs
            idx = df.index[changed]
            if len(idx):
                self.audit.record(sensor, df, idx, c, rule,
                                  old[changed], (old[changed] if note_only else newv[changed]))
                if not self.audit.dry_run:
                    if not note_only:
                        df.loc[idx, c] = newv[changed]
                    fc = c + "_flags"
                    if fc in df.columns:
                        df.loc[idx, fc] = df.loc[idx, fc].to_numpy() | (1 << bit)
                total += len(idx)
        return total


def rows_for_hours(df, hours, default=3):
    """Convert a time span to a row count using the median interval of each period,
    so a window means the same duration in hourly and per-minute data."""
    med = df.groupby("period")["interval_min"].transform("median")
    n = (hours * 60 / med.replace(0, np.nan)).round()
    n = n.fillna(default).clip(lower=3)
    # one window size per period; take the period's own value
    return int(max(3, n.median()))

## 1. Cleaning stages

Each stage is a function that mutates through `Fixer`. The order matters: structural
fixes (unmeasured sensors, reversed meter) come before any statistics, because every
later threshold is computed on the corrected values.

In [ ]:
"""Cleaning stages. Each takes the dict of sensor frames and mutates through Fixer."""
import re
import numpy as np, pandas as pd


# ---------------------------------------------------------------- 1. LOAD
def load(sheets, file_path, fx, info, original):
    """sheets: {name: DataFrame}. Returns {name: prepared DataFrame}."""
    def zone_of(name):
        for text in (name, file_path):
            m = re.search(CONFIG["zone_pattern"], str(text).upper())
            if m:
                return m.group(1)
        return "ZONE"

    data = {}
    for name, df in sheets.items():
        df = df.copy()
        df["excel_row"] = df.index + 2
        df["Timestamp"] = pd.to_datetime(df["Timestamp"], errors="coerce")
        n0 = len(df)
        df = (df.dropna(subset=["Timestamp"]).sort_values("Timestamp")
                .drop_duplicates("Timestamp").reset_index(drop=True))
        for c in ELEC + ENV:
            df[c] = pd.to_numeric(df[c], errors="coerce").astype(float)
        dt = df["Timestamp"].diff()
        df["interval_min"] = dt.dt.total_seconds() / 60
        df["period"] = (dt > pd.Timedelta(days=CONFIG["new_period_gap_days"])).cumsum() + 1
        df["freq"] = np.where(
            df.groupby("period")["interval_min"].transform("median") <= 5, "minute", "hour")
        Fixer.init_flags(df)
        original[name] = df[["Timestamp"] + ELEC + ENV].copy()
        info[name] = {"zone": zone_of(name),
                      "device": str(df["E Device type"].dropna().iloc[0]),
                      "meters": int(df["Number of Edevices"].dropna().iloc[0]),
                      "sign": 1, "measured": True, "rows": len(df),
                      "dropped_rows": n0 - len(df), "periods": int(df["period"].nunique())}
        data[name] = df
    return data


# ---------------------------------------------------------------- 2. UNMEASURED SENSORS
def drop_unmeasured(data, info):
    for s, df in data.items():
        if (df[ELEC].fillna(0) == 0).all().all():
            info[s]["measured"] = False
    return [s for s in data if info[s]["measured"]]


# ---------------------------------------------------------------- 3. INVERTED METER
def fix_inverted(data, measured, info, fx):
    inverted = []
    for s in measured:
        df = data[s]
        med = df.loc[df["Pa"] != 0, "Pa"].median()
        if pd.notna(med) and med < 0:
            for c in ELEC:
                flipped = -df[c]
                fx.apply(s, df, df[c].notna(), c, "sign_inverted", new=flipped)
            info[s]["sign"] = -1
            inverted.append(s)
    return inverted


# ---------------------------------------------------------------- 4. EMPTY READINGS
def blank_all_zero(data, measured, fx):
    for s in measured:
        df = data[s]
        fx.apply(s, df, (df[ELEC] == 0).all(axis=1), ELEC, "all_zero_reading")


def blank_bad_counters(data, measured, info, fx):
    for s in measured:
        df = data[s]
        for c in ["Ea", "Er"]:
            fx.apply(s, df, df[c] == 0, c, "counter_zero")
            fx.apply(s, df, df[c] < 0, c, "counter_negative")


# ---------------------------------------------------------------- 5. CORRUPTED VALUES
def blank_corrupted(data, measured, fx):
    for s in measured:
        df = data[s]
        for c in ["Pa", "W", "I"]:
            a = df[c].abs()
            typical = a[a > 0].median()
            limit = max(CONFIG["corrupted_x_median"] * (typical or 1),
                        CONFIG["corrupted_x_p999"] * a.quantile(0.999))
            fx.apply(s, df, a > limit, c, "corrupted_value")


# ---------------------------------------------------------------- 6. NEGATIVE VALUES
def fix_negatives(data, measured, fx):
    """W is ACTIVE power, so on a consuming load every negative Pa, W or I is an
    error -- not only those where another column is also negative."""
    def upper_limit(v):
        v = v[v > 0].dropna()
        if len(v) < 20:
            return np.inf
        q1, q3 = v.quantile([0.25, 0.75])
        return q3 + CONFIG["tukey_k"] * max(q3 - q1, 1e-9)

    for s in measured:
        df = data[s]
        for c in ["Pa", "W", "I"]:
            neg = df[c] < 0
            if not neg.any():
                continue
            hi = df.groupby("freq")[c].transform(lambda v: upper_limit(v))
            recoverable = neg & (df[c].abs() <= hi)
            fx.apply(s, df, recoverable, c, "negative_recovered", new=df[c].abs())
            fx.apply(s, df, neg & ~recoverable, c, "negative_nulled")


# ---------------------------------------------------------------- 7. COUNTER STATES
def classify_counter(df, col, power):
    """Label each counter reading: '', partial, spike, reset or jump."""
    state = pd.Series("", index=df.index, dtype=object)
    p = power.abs()
    p99 = p.quantile(0.99) if p.notna().any() else 0
    pmax = p.quantile(0.999) * 1.5 if p.notna().any() else np.inf
    ptyp = p.quantile(0.90) if p.notna().any() else np.inf
    nominal_h = df.groupby("period")["interval_min"].transform("median").fillna(60) / 60
    tol = np.maximum(CONFIG["counter_tolerance"], 2 * p99 * nominal_h)
    window = pd.Timedelta(hours=CONFIG["drop_return_hours"]).value
    for _, idx in df.groupby("period").groups.items():
        s = df.loc[idx, col].dropna()
        if len(s) < 2:
            continue
        v, rows = s.to_numpy(), s.index.to_numpy()
        t = df.loc[rows, "Timestamp"].to_numpy().astype("datetime64[ns]").astype(np.int64)
        tl = tol.loc[rows].to_numpy()
        prev, t_prev, i = v[0], t[0], 1
        while i < len(v):
            d, hours = v[i] - prev, (t[i] - t_prev) / 3.6e12
            if d < -tl[i]:
                end = np.searchsorted(t, t[i] + window, side="right")
                back = np.nonzero(v[i + 1:end] >= prev - tl[i])[0]
                if len(back):
                    j = i + 1 + back[0]
                    state[rows[i:j]] = "partial"
                    i = j
                    continue
                state[rows[i]] = "reset"
                prev, t_prev = v[i], t[i]
            elif d > CONFIG["jump_margin"] * (
                    pmax if hours <= CONFIG["max_power_energy_gap_h"] else ptyp
                 ) * max(hours, 1 / 60) + tl[i]:
                after = v[i + 1:i + 1 + CONFIG["jump_confirm_readings"]]
                if len(after) and np.any(np.abs(after - v[i]) <=
                                         tl[i] + pmax * CONFIG["jump_confirm_readings"] * max(hours, 1 / 60)):
                    state[rows[i]] = "jump"
                    prev, t_prev = v[i], t[i]
                else:
                    state[rows[i]] = "spike"
            else:
                prev, t_prev = v[i], t[i]
            i += 1
    return state


def clean_counters(data, measured, fx, events):
    """Ea pairs with Pa (apparent), Er pairs with W (active)."""
    for s in measured:
        df = data[s]
        for c, pw in [("Ea", df["Pa"]), ("Er", df["W"])]:
            st = classify_counter(df, c, pw)
            df[f"state_{c}"] = st
            fx.apply(s, df, st == "partial", c, "counter_partial")
            fx.apply(s, df, st == "spike", c, "counter_spike")
            # resets and jumps keep their value: they are a new valid level
            fx.apply(s, df, st == "reset", c, "counter_reset", note_only=True)
            fx.apply(s, df, st == "jump", c, "counter_jump", note_only=True)
            for i in df.index[st.isin(["reset", "jump"])]:
                events.append({"sensor": s, "timestamp": df.at[i, "Timestamp"],
                               "event": st[i], "column": c, "excel_row": df.at[i, "excel_row"]})


def blank_partial_power(data, measured, info, fx):
    for s in measured:
        df = data[s]
        if info[s]["meters"] < 2:
            continue
        n = rows_for_hours(df, CONFIG["partial_ref_window_h"])
        ref = df["Pa"].rolling(n, center=True, min_periods=5).median()
        partial = df["state_Ea"].isin(["partial", "spike"]) & (df["Pa"] < CONFIG["partial_ratio"] * ref)
        fx.apply(s, df, partial, ["Pa", "W", "I"], "partial_power")


# ---------------------------------------------------------------- 8. MISLEADING ZEROS
def blank_false_zeros(data, measured, fx):
    for s in measured:
        df = data[s]
        ea_up = df["Ea"].ffill().diff() > 0
        er_up = df["Er"].ffill().diff() > 0
        fx.apply(s, df, (df["Pa"] == 0) & ea_up, "Pa", "zero_is_missing")
        fx.apply(s, df, (df["W"] == 0) & er_up, "W", "zero_is_missing")
        fx.apply(s, df, (df["I"] == 0) & (df["Pa"].abs() > CONFIG["min_power_for_current_W"]),
                 "I", "zero_is_missing")


# ---------------------------------------------------------------- 9. OUTLIERS
def handle_outliers(data, measured, fx):
    def limits(v):
        v = v.dropna(); v = v[v != 0]
        if len(v) < 20:
            return -np.inf, np.inf
        q1, q3 = v.quantile([0.25, 0.75]); iqr = max(q3 - q1, 1e-9)
        return q1 - CONFIG["tukey_k"] * iqr, q3 + CONFIG["tukey_k"] * iqr

    for s in measured:
        df = data[s]
        very_high = {c: df[c].abs() > df.groupby("freq")[c].transform(
                        lambda v: v.abs().quantile(0.99)) for c in ["Pa", "W", "I"]}
        for c, confirm_with in [("Pa", ["I", "W"]), ("I", ["Pa"]), ("W", ["Pa", "I"])]:
            out = pd.Series(False, index=df.index)
            for f, g in df.groupby("freq"):
                lo, hi = limits(g[c])
                out.loc[g.index] = (g[c] != 0) & (g[c].abs() > hi)
            checks = [very_high[x] for x in confirm_with]
            if c == "W":        # active power confirmed by the active counter
                hours = df["interval_min"] / 60
                implied = (df["Er"].diff() / hours).where(df["Er"].notna() & df["Er"].shift().notna())
                checks.append((implied - df["W"]).abs() <= 0.5 * df["W"].abs())
            if c == "Pa":       # apparent power confirmed by the apparent counter
                hours = df["interval_min"] / 60
                implied = (df["Ea"].diff() / hours).where(df["Ea"].notna() & df["Ea"].shift().notna())
                checks.append((implied - df["Pa"]).abs() <= 0.5 * df["Pa"].abs())
            confirmed = out & np.logical_or.reduce(checks)
            fx.apply(s, df, confirmed, c, "outlier_kept", note_only=True)
            fx.apply(s, df, out & ~confirmed, c, "outlier_nulled")


def flag_power_factor(data, measured, fx):
    """P <= S is a hard law per meter, but these columns are sums over many meters and
    a small share of rows breaks it. The pipeline cannot know which of the two is
    wrong, so the rows are flagged and left untouched for the modeller to judge."""
    for s in measured:
        df = data[s]
        bad = (df["W"] > df["Pa"] * 1.01) & df["W"].notna() & df["Pa"].notna()
        fx.apply(s, df, bad, ["W", "Pa"], "pf_implausible", note_only=True)


# ---------------------------------------------------------------- 10. ENVIRONMENT
def build_environment(data, info, fx):
    """One environmental table per zone; cleaned once, written back to every sheet."""
    environment = {}
    for zone in sorted({info[s]["zone"] for s in data}):
        sheets = [s for s in data if info[s]["zone"] == zone]
        ref = max(sheets, key=lambda s: len(data[s]))
        everything = pd.concat([data[s][["Timestamp", "excel_row"] + ENV].assign(_ref=(s == ref))
                                for s in sheets])
        e = (everything.sort_values(["Timestamp", "_ref"], ascending=[True, False])
                       .drop_duplicates("Timestamp").drop(columns="_ref").reset_index(drop=True))
        dt = e["Timestamp"].diff()
        e["interval_min"] = dt.dt.total_seconds() / 60
        e["period"] = (dt > pd.Timedelta(days=CONFIG["new_period_gap_days"])).cumsum() + 1
        Fixer.init_flags(e, ENV)
        environment[zone] = e
        for s in sheets:
            data[s] = data[s].drop(columns=ENV)
    return environment


def clean_environment(environment, fx):
    for zone, e in environment.items():
        tag = f"zone {zone}"
        absent = [c for c in ENV if (e[c].fillna(0) == 0).all()]
        if absent:
            fx.apply(tag, e, pd.Series(True, index=e.index), absent, "env_sensor_absent")
        missing = (e["Hum"] == 0) & (e["Temp"] == 0)
        fx.apply(tag, e, missing, ENV, "env_reading_missing")
        for c in ["Temp", "Hum"]:
            fx.apply(tag, e, e[c] == 0, c, "env_reading_missing")
        lo, hi = CONFIG["temp_physical"]
        fx.apply(tag, e, (e["Temp"] < lo) | (e["Temp"] > hi), "Temp", "temp_impossible")
        n = rows_for_hours(e, CONFIG["hum_median_window_h"])
        med = e["Hum"].rolling(n, center=True, min_periods=5).median()
        fx.apply(tag, e, ((e["Hum"] - med).abs() > CONFIG["hum_jump"])
                         | (e["Hum"] <= 0) | (e["Hum"] > 100), "Hum", "hum_isolated")
        fx.apply(tag, e, e["Pres"].notna() & ~e["Pres"].isin([0, 1]), "Pres", "pres_not_binary")


# ---------------------------------------------------------------- 11. SHORT GAPS
def interpolate_short_gaps(data, measured, environment, fx):
    def fillable_mask(s, max_len):
        empty = s.isna()
        groups = (empty != empty.shift()).cumsum()
        length = empty.groupby(groups).transform("sum")
        return empty & (length <= max_len)

    for s in measured:
        df = data[s]
        k = rows_for_hours(df, CONFIG["max_interp_gap_h"])
        for c in ["Pa", "W", "I"]:
            out = pd.Series(np.nan, index=df.index)
            for _, idx in df.groupby("period").groups.items():
                seg = df.loc[idx, c]
                out.loc[idx] = seg.interpolate(limit_area="inside")
            m = fillable_mask(df[c], k) & out.notna()
            fx.apply(s, df, m, c, "interpolated", new=out)

    for zone, e in environment.items():
        k = rows_for_hours(e, CONFIG["max_interp_gap_h"])
        for c in ["Hum", "Temp"]:
            out = pd.Series(np.nan, index=e.index)
            for _, idx in e.groupby("period").groups.items():
                out.loc[idx] = e.loc[idx, c].interpolate(limit_area="inside")
            m = fillable_mask(e[c], k) & out.notna()
            fx.apply(f"zone {zone}", e, m, c, "interpolated", new=out)


# ---------------------------------------------------------------- 12. ENERGY
def interval_energy(df, col, power):
    """Energy consumed in each interval, plus a provenance label per row:
       counter   - the genuine difference between two valid counter readings
       spread    - that difference shared across intermediate rows, weighted by power
       power     - estimated as mean power x time (reset, jump, start of a period)
       unknown   - neither available
    """
    hours = df["interval_min"].fillna(0) / 60
    same = df["period"] == df["period"].shift()
    e_pow = ((power + power.shift()) / 2 * hours).where(
        same & (df["Timestamp"].diff() <= pd.Timedelta(hours=CONFIG["max_power_energy_gap_h"])))
    valid = df[col].notna().to_numpy()
    pos = np.arange(len(df), dtype=float)
    nxt = pd.Series(np.where(valid, pos, np.nan), index=df.index).bfill()
    prv = pd.Series(np.where(valid, pos, np.nan), index=df.index).ffill().shift()
    b = np.nonzero(valid & prv.notna().to_numpy())[0]
    a = prv.to_numpy()[b].astype(int)
    v, per, st = df[col].to_numpy(), df["period"].to_numpy(), df[f"state_{col}"].to_numpy()
    d = v[b] - v[a]
    ok = (d >= 0) & (per[a] == per[b]) & ~np.isin(st[b], ["reset", "jump"])
    inc = pd.Series(np.nan, index=df.index)
    inc.iloc[b[ok]] = d[ok]
    weight = e_pow.clip(lower=0).fillna(0)
    total = weight.groupby(nxt).transform("sum")
    n = weight.groupby(nxt).transform("count")
    share = np.where(total > 0, weight / total.replace(0, np.nan), 1 / n)
    block_size = pd.Series(1, index=df.index).groupby(nxt).transform("size")
    energy = pd.Series(nxt.map(inc).to_numpy() * share, index=df.index)

    source = pd.Series("unknown", index=df.index, dtype=object)
    source[energy.notna() & (block_size == 1)] = "counter"
    source[energy.notna() & (block_size > 1)] = "spread"
    from_power = energy.isna() & e_pow.notna()
    energy[from_power] = e_pow[from_power]
    source[from_power] = "power"
    first_row = df["period"] != df["period"].shift()
    source[first_row & energy.isna()] = "period_start"
    return energy, source


def build_energy(data, measured):
    for s in measured:
        df = data[s]
        # ACTIVE energy (the forecasting target): Er with W
        df["energy_Wh"], df["energy_source"] = interval_energy(df, "Er", df["W"])
        # APPARENT energy, kept for power-factor work: Ea with Pa
        df["apparent_energy_VAh"], _ = interval_energy(df, "Ea", df["Pa"])
        for c, en in [("Er", "energy_Wh"), ("Ea", "apparent_energy_VAh")]:
            consumed = df.groupby("period")[en].transform(lambda x: x.fillna(0).cumsum())
            start = df.groupby("period")[c].transform("first").clip(lower=0)
            df[f"{c}_clean"] = start + consumed

## 2. Assembly and checks

The output keeps the original 12 columns, in order, then adds:

| Column | Meaning |
|---|---|
| `W_flags`, `Pa_flags`, `I_flags`, `Ea_flags`, `Er_flags`, `Lum_flags`, `Hum_flags`, `Temp_flags`, `Pres_flags` | bitmask of the rules that touched that cell (0 = untouched) |
| `energy_Wh` | **active** energy consumed in the interval — the forecasting target |
| `energy_source` | `counter` (measured), `spread` (counter difference shared by power), `power` (power x time), `period_start`, `unknown` |
| `apparent_energy_VAh` | apparent energy, for power-factor work |

The final energy check is deliberately **not circular**: it uses only rows where the
energy came straight from a counter difference, so power played no part in building
the value. It tests the median ratio (bias), not the scatter, because an interval
average and an end-of-interval snapshot legitimately differ on a cycling load.

In [ ]:
"""Assemble the output, run the checks, write the files."""
import re
import numpy as np, pandas as pd


def run(sheets, file_path, dry_run=False):
    run_id = make_run_id(CONFIG)
    audit = Audit(run_id, dry_run=dry_run)
    fx = Fixer(audit)
    info, original, events = {}, {}, []

    data = load(sheets, file_path, fx, info, original)
    measured = drop_unmeasured(data, info)
    inverted = fix_inverted(data, measured, info, fx)
    blank_all_zero(data, measured, fx)
    blank_bad_counters(data, measured, info, fx)
    blank_corrupted(data, measured, fx)
    fix_negatives(data, measured, fx)
    clean_counters(data, measured, fx, events)
    blank_partial_power(data, measured, info, fx)
    blank_false_zeros(data, measured, fx)
    handle_outliers(data, measured, fx)
    flag_power_factor(data, measured, fx)
    environment = build_environment(data, info, fx)
    clean_environment(environment, fx)
    interpolate_short_gaps(data, measured, environment, fx)
    build_energy(data, measured)

    dataset = assemble(data, measured, environment, info)
    checks, energy_check, quality = run_checks(dataset, data, measured, environment, info)
    return dict(run_id=run_id, dataset=dataset, data=data, environment=environment,
                info=info, inverted=inverted, events=pd.DataFrame(events),
                audit=audit, checks=(checks, energy_check), quality=quality,
                original=original, measured=measured)


def assemble(data, measured, environment, info):
    pieces = []
    for s in measured:
        df = data[s]
        t = df[["Timestamp", "W", "I", "Pa"]].copy()
        t["Ea"] = df["Ea_clean"]
        t["Er"] = df["Er_clean"]
        for c in ["Pa", "W", "I", "Ea", "Er"]:
            t[c + "_flags"] = df[c + "_flags"]
        t["energy_Wh"] = df["energy_Wh"]
        t["energy_source"] = df["energy_source"]
        t["apparent_energy_VAh"] = df["apparent_energy_VAh"]
        n_before = len(t)
        env = environment[info[s]["zone"]]
        envcols = ["Timestamp"] + ENV + [c + "_flags" for c in ENV]
        t = t.merge(env[envcols], on="Timestamp", how="left")
        assert len(t) == n_before, f"{s}: environmental join changed the row count"
        t.insert(0, "sensor", s)
        t["E Device type"] = info[s]["device"]
        t["Number of Edevices"] = info[s]["meters"]
        pieces.append(t)
    dataset = pd.concat(pieces, ignore_index=True)
    dataset = dataset[["sensor"] + COLUMNS + EXTRA_COLS]
    num = [c for c in dataset.select_dtypes("number").columns
           if c not in FLAG_COLS + ["Number of Edevices"]]
    dataset[num] = dataset[num].round(4)
    for c in FLAG_COLS:
        dataset[c] = dataset[c].fillna(0).astype("int64")
    return dataset


def run_checks(dataset, data, measured, environment, info):
    res = []
    def chk(name, ok, detail=""):
        res.append({"check": name, "result": "PASS" if ok else "FAIL", "detail": str(detail)})

    chk("Original columns present, in order, before the new ones",
        list(dataset.columns[1:1 + len(COLUMNS)]) == COLUMNS)
    chk("Provenance columns present", all(c in dataset.columns for c in EXTRA_COLS))
    chk("One row per original timestamp",
        all((dataset["sensor"] == s).sum() == info[s]["rows"] for s in measured),
        f"{len(dataset):,} rows")
    dup = dataset.duplicated(["sensor", "Timestamp"]).sum()
    srt = all(g["Timestamp"].is_monotonic_increasing for _, g in dataset.groupby("sensor"))
    chk("Timestamps unique and sorted", dup == 0 and srt, f"{dup} duplicates")

    # monotonic counters, using the period boundaries the pipeline itself stored,
    # not a re-derived threshold
    mono = True
    for s in measured:
        per = data[s]["period"].to_numpy()
        g = dataset[dataset["sensor"] == s]
        for c in ["Ea", "Er"]:
            v = g[c].to_numpy()
            d = np.diff(v)
            same = per[1:] == per[:-1]
            mono &= bool(np.all(d[same & ~np.isnan(d)] >= -1e-6))
    chk("Ea and Er never decrease inside a collection period", mono)

    chk("No negative Pa, W or I",
        bool((dataset[["Pa", "W", "I"]].dropna() >= 0).all().all()),
        f"min W {dataset['W'].min()}, min Pa {dataset['Pa'].min()}")
    # P <= S holds per meter; these columns are sums over up to 92 meters and a small
    # share of rows breaks it in the SOURCE data. Reported and flagged, never a blocker.
    viol = dataset.dropna(subset=["W", "Pa"]).eval("W > Pa * 1.01")
    worst = (dataset.assign(_v=dataset.eval("W > Pa * 1.01"))
             .groupby("sensor")["_v"].mean().sort_values(ascending=False))
    chk("NOTE: rows where active power exceeds apparent power (flagged, not corrected)",
        True, f"{viol.sum():,} rows ({viol.mean()*100:.2f}%); worst "
              f"{worst.index[0]} {worst.iloc[0]*100:.1f}%")
    chk("Lum >= 0", bool((dataset["Lum"].dropna() >= 0).all()))
    chk("Hum within (0, 100]", bool(dataset["Hum"].dropna().between(0, 100, inclusive="right").all()))
    lo, hi = CONFIG["temp_physical"]
    chk("Temp within the physical range and never exactly 0",
        bool(dataset["Temp"].dropna().between(lo, hi).all() and (dataset["Temp"].dropna() != 0).all()),
        f"min {dataset['Temp'].min()}, max {dataset['Temp'].max()}")
    chk("Pres only 0 or 1", bool(dataset["Pres"].dropna().isin([0, 1]).all()))
    chk("No infinite values",
        bool(np.isfinite(dataset.select_dtypes("number").fillna(0).to_numpy()).all()))

    # every flagged cell is explained by the audit table, and vice versa
    chk("energy_source labels are known",
        set(dataset["energy_source"].dropna()) <= {"counter", "spread", "power",
                                                   "period_start", "unknown"},
        sorted(set(dataset["energy_source"].dropna())))

    # NON-CIRCULAR energy check.
    # Only rows whose energy came straight from a counter difference are used
    # (source == "counter"), so power never took part in building the value.
    # We test the BIAS (median of implied/W), not the scatter: an interval average
    # and an end-of-interval snapshot legitimately differ on a cycling load.
    # Evaluated per sampling regime, because the 2025 counters are rounded to whole
    # Wh and a per-minute difference is then mostly quantisation.
    rows = []
    for s in measured:
        df = data[s]
        for freq, g in df.groupby("freq"):
            m = ((g["energy_source"] == "counter") & g["W"].notna() & (g["W"] > 1)
                 & (g["interval_min"] > 0))
            if m.sum() < 200:
                continue
            implied = g.loc[m, "energy_Wh"] / (g.loc[m, "interval_min"] / 60)
            ratio = implied / g.loc[m, "W"]
            scatter = (implied - g.loc[m, "W"]).abs() / g.loc[m, "W"].clip(lower=1)
            rows.append({"sensor": s, "regime": freq, "rows compared": int(m.sum()),
                         "median W": round(float(g.loc[m, "W"].median()), 1),
                         "bias (median ratio)": round(float(ratio.median()), 3),
                         "scatter (median |err|) %": round(float(scatter.median()) * 100, 1)})
    energy_check = pd.DataFrame(rows)

    # Per-sensor verdict on whether the active counter can be trusted as the target.
    # A sensor is never dropped and the run is never blocked: the verdict travels with
    # the data so the modeller can exclude or down-weight it deliberately.
    quality = []
    for s in measured:
        h = energy_check[(energy_check["sensor"] == s) & (energy_check["regime"] == "hour")]
        if h.empty:
            verdict, why = "no_hourly_data", "too few comparable hourly rows"
        else:
            bias, mw = float(h["bias (median ratio)"].iloc[0]), float(h["median W"].iloc[0])
            if mw < 5:
                verdict, why = "below_resolution", f"median {mw:.1f} W, near the counter step"
            elif 0.9 <= bias <= 1.1:
                verdict, why = "ok", f"bias {bias:.3f}"
            else:
                verdict, why = "counter_biased", f"counter runs at {bias:.3f} of power"
        quality.append({"sensor": s, "energy_verdict": verdict, "reason": why})
    quality = pd.DataFrame(quality)

    good = (quality["energy_verdict"] == "ok").sum()
    chk("Active energy from the counter is unbiased against active power (hourly)",
        True, f"{good}/{len(quality)} sensors ok")
    for v in ["counter_biased", "below_resolution", "no_hourly_data"]:
        names = quality.loc[quality["energy_verdict"] == v, "sensor"].tolist()
        if names:
            chk(f"NOTE: sensors marked '{v}' - use energy_Wh there with care",
                True, ", ".join(names))
    if (energy_check["regime"] == "minute").any():
        chk("NOTE: per-minute counters are rounded to whole Wh; "
            "aggregate to >=15 min before using differences", True,
            f"{(energy_check['regime'] == 'minute').sum()} series affected")
    return pd.DataFrame(res), energy_check, quality

## 3. Run it

Upload the workbook, run the pipeline, inspect the checks.

In [ ]:
try:
    from google.colab import files
    print("Choose the Excel file to clean:")
    uploaded = files.upload()
    FILE_PATH = next(iter(uploaded))
except ImportError:
    FILE_PATH = input("Path of the Excel file to clean: ").strip()

sheets = pd.read_excel(FILE_PATH, sheet_name=None)
missing = {n: [c for c in COLUMNS if c not in d.columns] for n, d in sheets.items()}
assert not any(missing.values()), f"Missing columns: { {k:v for k,v in missing.items() if v} }"
print(f"{len(sheets)} sheets, {sum(len(d) for d in sheets.values()):,} rows")

result = run(sheets, FILE_PATH)
dataset, audit = result["dataset"], result["audit"]
checks, energy_check = result["checks"]
quality = result["quality"]
print(f"\nrun_id: {result['run_id']}")
print("Sensors left out (no measurement):",
      [s for s in result["info"] if not result["info"][s]["measured"]] or "none")
print("Meters wired in reverse:", result["inverted"] or "none")
display(checks)
display(quality)
display(energy_check)

## 4. What was changed

`audit.parquet` has one row per changed cell. These views answer the three questions
people actually ask: how much was touched, by which rule, and what happened to this
particular value.

In [ ]:
A = audit.frame()
print(f"{len(A):,} cell-level changes\n")

print("By rule:")
display(A.groupby("rule").agg(cells=("rule", "size"),
                              sensors=("sensor", "nunique")).sort_values("cells", ascending=False))

print("\nShare of each column that was touched:")
display(pd.DataFrame([{"column": c,
                       "touched": int((dataset[c + "_flags"] != 0).sum()),
                       "% of rows": round((dataset[c + "_flags"] != 0).mean() * 100, 1)}
                      for c in FLAGGED]).set_index("column").T)

print("\nProvenance of the forecasting target (energy_Wh):")
display(dataset["energy_source"].value_counts(dropna=False).rename("rows").to_frame()
        .assign(**{"%": lambda d: (d["rows"] / len(dataset) * 100).round(1)}))

print("\nFull history of one cell:")
if len(A):
    ex = A.iloc[0]
    display(A[(A.sensor == ex.sensor) & (A.excel_row == ex.excel_row) &
              (A.column == ex.column)][["rule", "old_value", "new_value"]])

## 5. Decoding a flag

Any flagged cell can be read back without the audit table: `Pa_flags & 128` is true
for every active-power value whose sign was corrected. This helper turns a bitmask
into rule names.

In [ ]:
def decode(mask):
    return [v["rule"] for b, v in RULES.items() if int(mask) >> b & 1]

# the codebook now carries the justification for every rule
pd.set_option("display.max_colwidth", 70)
display(rules_table()[["bit", "flag_value", "rule", "what_detected", "action"]])
print("\nFull justification for one rule:")
display(rules_table().query("rule == 'counter_partial'").T)

sample = dataset[dataset["W_flags"] != 0].head(5)[["sensor", "Timestamp", "W", "W_flags"]].copy()
sample["rules"] = sample["W_flags"].map(lambda m: ", ".join(decode(m)))
display(sample)

## 6. Save

Six CSV files. `clean_Storey1_<ZONE>.csv` is the deliverable: every sensor of the zone
in one table, with a `sensor` column telling them apart. The other five are the audit
trail and are what the report's methodology section is built from.

Reading it back in pandas:

```python
df = pd.read_csv("clean_Storey1_NW.csv", parse_dates=["Timestamp"])
for c in [x for x in df.columns if x.endswith("_flags")]:
    df[c] = df[c].fillna(0).astype("int64")      # CSV loses the integer type
```

That cast matters: without it the flag columns come back as floats and the bitwise
tests (`df.W_flags & 4`) fail.

In [ ]:
import os

ZONE = next(iter(result["info"].values()))["zone"]
OUT = f"clean_Storey1_{ZONE}.csv"

# the meters report 3 decimals at best and the 2025 values are integer-rounded,
# so 2 decimals keeps every digit that carries information
num = [c for c in dataset.select_dtypes("number").columns
       if not c.endswith("_flags") and c != "Number of Edevices"]
out = dataset.copy()
out[num] = out[num].round(2)
out.to_csv(OUT, index=False)

A.to_csv("audit.csv.gz", index=False, compression="gzip")
audit.summary().to_csv("correction_log.csv", index=False)
rules_table().to_csv("rules.csv", index=False)
result["events"].to_csv("counter_events.csv", index=False)
quality.to_csv("sensor_quality.csv", index=False)
pd.DataFrame([{"run_id": result["run_id"], "file": FILE_PATH,
               **{f"cfg.{k}": str(v) for k, v in CONFIG.items()}}]).to_csv("run_config.csv", index=False)

FILES = [OUT, "audit.csv.gz", "correction_log.csv", "rules.csv",
         "counter_events.csv", "sensor_quality.csv", "run_config.csv"]
for f in FILES:
    print(f"{f:28s} {os.path.getsize(f)/1e6:8.2f} MB")

assert (checks["result"] == "PASS").all(), "A check failed - see the table above."
print(f"\n{len(out):,} rows written. All checks passed.")

try:
    from google.colab import files
    for f in FILES:
        files.download(f)
except ImportError:
    pass